In [1]:
import numpy as np
import networkx as nx
import itertools as it
import random as rd
import pickle as pk
import pandas as pd
from collections import (defaultdict,Counter)
import matplotlib.pyplot as plt
import scipy  
from scipy import stats


In [2]:
def overlap_jaccard(list1,list2):
    intersction_term= len(set(list1) & set(list2))
    denominator = len(set(list1).union(set(list2)))
    overlap_jaccard_coeff = intersction_term/denominator
    return overlap_jaccard_coeff

def overlap_genelists(lstA, lstB, background):
    import scipy.stats as stats
    """
    Accepts to lists
    M is the population size (previously N)
    n is the number of successes in the population
    N is the sample size (previously n)
    x is still the number of drawn “successes”
    """
    setA= set(lstA)
    setB= set(lstB)
    M= background #total number of genes
    n= len(setA)
    N= len(setB)
    x= len(setA.intersection(setB))


    return str(stats.hypergeom.sf(x-1, M, n, N))



#Here we define the adjustment
from statsmodels.sandbox.stats.multicomp import multipletests
def fdr_adjustment(list_of_pvals,alpha):    
    return multipletests(list_of_pvals,alpha=alpha,method='fdr_bh')[1] #the benjamin hochberg method is used

In [3]:
#Let's import the ICD-gene associations
with open('output/allsources_icd_gene_dict_unified.pickle', 'rb') as handle:
    allsources_icd_gene_dict_unified = pk.load(handle)

In [12]:
G_elma_icd_female_df = pd.read_csv('output/G_elma_icd_female_df.tsv', sep="\t",index_col=0)
G_elma_icd_male_df = pd.read_csv('output/G_elma_icd_male_df.tsv', sep="\t",index_col=0)

In [14]:
G_elma_icd_male_df

,Dis A,Dis B
0,K35,K37
1,K35,K56
2,K35,K65
3,K37,K56
4,K37,K65
...,...,...
1276,I70,I73
1277,J91,J94
1278,C80,D63
1279,D31,H04


In [73]:
with open('output/female_trajectories_dis_sort_bytime_grouped_all_add_dict.pickle', 'rb') as handle:
    female_trajectories_dis_sort_bytime_grouped_all_add_dict = pk.load(handle)

with open('output/female_trajectories_dict_filtered.pickle', 'rb') as handle:
    female_trajectories_dict_filtered = pk.load(handle)

with open('output/female_trajectories_dict_filtered_unique.pickle', 'rb') as handle:
    female_trajectories_dict_filtered_unique = pk.load(handle)


with open('output/male_trajectories_dis_sort_bytime_grouped_all_add_dict.pickle', 'rb') as handle:
    male_trajectories_dis_sort_bytime_grouped_all_add_dict = pk.load(handle)

with open('output/male_trajectories_dict_filtered.pickle', 'rb') as handle:
    male_trajectories_dict_filtered = pk.load(handle)

with open('output/male_trajectories_dict_filtered_unique.pickle', 'rb') as handle:
    male_trajectories_dict_filtered_unique = pk.load(handle)
    

In [69]:
female_dis_pair_phase_dict = {}
for i,v in G_elma_icd_female_df.iterrows():
    dis_a = v['Dis A']
    dis_b = v['Dis B']
    phase_dict = {}
    trj_list = []
    # Step 1: Identify trajectories containing both diseases
    for trj, dislist in female_trajectories_dict_filtered_unique.items():
        if dis_a in dislist and dis_b in dislist:
            trj_list.append(trj)
    cl_trj_list = []     
    for trj in trj_list:
        if "_" not in trj:
            cl_trj_list.append(trj)
        else:
            for cl_trj in trj.split("_"):
                cl_trj_list.append(cl_trj)
                
    
    same_time = []
    a_previous_b = []
    b_previous_a = []
    
    # Step 2: Analyze the trajectories to determine the relative occurrence of the diseases
    for trj in cl_trj_list:
        ph_trj_dict = female_trajectories_dis_sort_bytime_grouped_all_add_dict[int(trj)]
        found_same_time = False
        found_a_before_b = False
        found_b_before_a = False
    
        # Iterate over the stages in order
        for ph1, dislist1 in ph_trj_dict.items():
            # Check if both diseases occur in the same stage
            if dis_a in dislist1 and dis_b in dislist1:
                same_time.append(trj)
                found_same_time = True
                break  # No need to check further if they're in the same stage
    
        # If not same_time, check for a before b and b before a
        if not found_same_time:
            for ph1, dislist1 in ph_trj_dict.items():
                if dis_a in dislist1:
                    # Check if b occurs in a later stage
                    for ph2 in range(ph1 + 1, len(ph_trj_dict) + 1):
                        if dis_b in ph_trj_dict[ph2]:
                            a_previous_b.append(trj)
                            found_a_before_b = True
                            break
                if dis_b in dislist1 and not found_a_before_b:
                    # Check if a occurs in a later stage
                    for ph2 in range(ph1 + 1, len(ph_trj_dict) + 1):
                        if dis_a in ph_trj_dict[ph2]:
                            b_previous_a.append(trj)
                            found_b_before_a = True
                            break
                # Stop checking once either condition is satisfied
                if found_a_before_b or found_b_before_a:
                    break
    
    # Step 3: Calculate proportions
    total_trajectories = len(cl_trj_list)
    same_time_proportion = len(set(same_time)) / total_trajectories if total_trajectories > 0 else 0
    a_before_b_proportion = len(set(a_previous_b)) / total_trajectories if total_trajectories > 0 else 0
    b_before_a_proportion = len(set(b_previous_a)) / total_trajectories if total_trajectories > 0 else 0
    phase_dict["A & B same time"]=same_time_proportion
    phase_dict["A before B"]=a_before_b_proportion
    phase_dict["B before A"]=b_before_a_proportion
    female_dis_pair_phase_dict[dis_a,dis_b] = phase_dict
    


In [70]:
with open('output/female_dis_pair_phase_dict.pickle', 'wb') as handle:
    pk.dump(female_dis_pair_phase_dict, handle, protocol=pk.HIGHEST_PROTOCOL)

In [75]:
male_dis_pair_phase_dict = {}
for i,v in G_elma_icd_male_df.iterrows():
    dis_a = v['Dis A']
    dis_b = v['Dis B']
    phase_dict = {}
    trj_list = []
    # Step 1: Identify trajectories containing both diseases
    for trj, dislist in male_trajectories_dict_filtered_unique.items():
        if dis_a in dislist and dis_b in dislist:
            trj_list.append(trj)
    cl_trj_list = []     
    for trj in trj_list:
        if "_" not in trj:
            cl_trj_list.append(trj)
        else:
            for cl_trj in trj.split("_"):
                cl_trj_list.append(cl_trj)
                
    
    same_time = []
    a_previous_b = []
    b_previous_a = []
    
    # Step 2: Analyze the trajectories to determine the relative occurrence of the diseases
    for trj in cl_trj_list:
        ph_trj_dict = male_trajectories_dis_sort_bytime_grouped_all_add_dict[int(trj)]
        found_same_time = False
        found_a_before_b = False
        found_b_before_a = False
    
        # Iterate over the stages in order
        for ph1, dislist1 in ph_trj_dict.items():
            # Check if both diseases occur in the same stage
            if dis_a in dislist1 and dis_b in dislist1:
                same_time.append(trj)
                found_same_time = True
                break  # No need to check further if they're in the same stage
    
        # If not same_time, check for a before b and b before a
        if not found_same_time:
            for ph1, dislist1 in ph_trj_dict.items():
                if dis_a in dislist1:
                    # Check if b occurs in a later stage
                    for ph2 in range(ph1 + 1, len(ph_trj_dict) + 1):
                        if dis_b in ph_trj_dict[ph2]:
                            a_previous_b.append(trj)
                            found_a_before_b = True
                            break
                if dis_b in dislist1 and not found_a_before_b:
                    # Check if a occurs in a later stage
                    for ph2 in range(ph1 + 1, len(ph_trj_dict) + 1):
                        if dis_a in ph_trj_dict[ph2]:
                            b_previous_a.append(trj)
                            found_b_before_a = True
                            break
                # Stop checking once either condition is satisfied
                if found_a_before_b or found_b_before_a:
                    break
    
    # Step 3: Calculate proportions
    total_trajectories = len(cl_trj_list)
    same_time_proportion = len(set(same_time)) / total_trajectories if total_trajectories > 0 else 0
    a_before_b_proportion = len(set(a_previous_b)) / total_trajectories if total_trajectories > 0 else 0
    b_before_a_proportion = len(set(b_previous_a)) / total_trajectories if total_trajectories > 0 else 0
    phase_dict["A & B same time"]=same_time_proportion
    phase_dict["A before B"]=a_before_b_proportion
    phase_dict["B before A"]=b_before_a_proportion
    male_dis_pair_phase_dict[dis_a,dis_b] = phase_dict
    


In [76]:
with open('output/male_dis_pair_phase_dict.pickle', 'wb') as handle:
    pk.dump(male_dis_pair_phase_dict, handle, protocol=pk.HIGHEST_PROTOCOL)

In [66]:
G_elma_icd_female_genetic_jaccard_dict = {}
G_elma_icd_female_genetic_pval_dict={}

for i,v in G_elma_icd_female_df.iterrows():
    dis_a = v['Dis A']
    dis_b = v['Dis B']
    genelist_a = allsources_icd_gene_dict_unified[dis_a]
    genelist_b = allsources_icd_gene_dict_unified[dis_b]
    ji_weight= overlap_jaccard(genelist_a,genelist_b)
    G_elma_icd_female_genetic_jaccard_dict[dis_a,dis_b] = ji_weight
    pval_ov=float(overlap_genelists(genelist_a,genelist_b,24008)) #24008 is the total number of genes
    G_elma_icd_female_genetic_pval_dict[dis_a,dis_b] = pval_ov


G_elma_icd_female_genetic_fdr_dict = {}
adj_pval=fdr_adjustment(list(G_elma_icd_female_genetic_pval_dict.values()),0.05)
pair_list=list(G_elma_icd_female_genetic_pval_dict.keys())
for i in range(len(adj_pval)):
    G_elma_icd_female_genetic_fdr_dict[pair_list[i]]=adj_pval[i]
        


In [71]:
with open('output/G_elma_icd_female_genetic_jaccard_dict.pickle', 'wb') as handle:
    pk.dump(G_elma_icd_female_genetic_jaccard_dict, handle, protocol=pk.HIGHEST_PROTOCOL)

with open('output/G_elma_icd_female_genetic_fdr_dict.pickle', 'wb') as handle:
    pk.dump(G_elma_icd_female_genetic_fdr_dict, handle, protocol=pk.HIGHEST_PROTOCOL)

In [77]:
G_elma_icd_male_genetic_jaccard_dict = {}
G_elma_icd_male_genetic_pval_dict={}

for i,v in G_elma_icd_male_df.iterrows():
    dis_a = v['Dis A']
    dis_b = v['Dis B']
    genelist_a = allsources_icd_gene_dict_unified[dis_a]
    genelist_b = allsources_icd_gene_dict_unified[dis_b]
    ji_weight= overlap_jaccard(genelist_a,genelist_b)
    G_elma_icd_male_genetic_jaccard_dict[dis_a,dis_b] = ji_weight
    pval_ov=float(overlap_genelists(genelist_a,genelist_b,24008)) #24008 is the total number of genes
    G_elma_icd_male_genetic_pval_dict[dis_a,dis_b] = pval_ov


G_elma_icd_male_genetic_fdr_dict = {}
adj_pval=fdr_adjustment(list(G_elma_icd_male_genetic_pval_dict.values()),0.05)
pair_list=list(G_elma_icd_male_genetic_pval_dict.keys())
for i in range(len(adj_pval)):
    G_elma_icd_male_genetic_fdr_dict[pair_list[i]]=adj_pval[i]
        


In [78]:
with open('output/G_elma_icd_male_genetic_jaccard_dict.pickle', 'wb') as handle:
    pk.dump(G_elma_icd_male_genetic_jaccard_dict, handle, protocol=pk.HIGHEST_PROTOCOL)

with open('output/G_elma_icd_male_genetic_fdr_dict.pickle', 'wb') as handle:
    pk.dump(G_elma_icd_male_genetic_fdr_dict, handle, protocol=pk.HIGHEST_PROTOCOL)